# Connecting to Alibre: every way

Several entry points bind a Python script to a running Alibre Design
instance. They run below from simplest to most explicit; pick whatever
fits your context.

**Prereq:** Alibre Design must be running.

## 1. `import alibrex`

The import verifies Alibre is running. If it's not, you get
an :class:`AlibreNotRunning` immediately. No further setup is needed.
This is the implicit "are we connected?" check at the top of every
script.

In [ ]:
import alibrex
alibrex.__version__

## 2. `connect()`: short alias

Returns the typed :class:`IADRoot` proxy. Use when you want explicit
root access for low-level calls like `root.Sessions`,
`root.NewObjectCollector()`, `root.MaterialLibraries`, etc.

In [ ]:
from alibrex import connect

In [ ]:
root = connect()
root.Version

Browse open sessions through the root:

In [ ]:
root.Sessions.Count

## 3. `connect_to_running_alibre()`: the long name

Identical to `connect()`. Use whichever name reads better in your
code. Both return the same cached proxy.

In [ ]:
from alibrex import connect_to_running_alibre

In [ ]:
connect_to_running_alibre() is connect()

## 4. `CurrentPart()`: when a part is open

The simplest path when you know the active document is a part: it
connects implicitly and narrows the active session to
:class:`IADPartSession` in one step. No root needed.

> **Prereq for this cell:** open any part in Alibre (File > New > Part).

In [ ]:
from alibrex import CurrentPart

In [ ]:
part = CurrentPart()
part.Name

Raises `RuntimeError` if no document is open *or* the active document isn't a part.

## 5. `CurrentAssembly()`: when an assembly is open

Same shape for assemblies. Narrows to :class:`IADAssemblySession`.

> **Prereq for this cell:** open any assembly in Alibre.

In [ ]:
from alibrex import CurrentAssembly

In [ ]:
asm = CurrentAssembly()
asm.Name

## 6. Strict checkers: `require_active_*`

When you already have an :class:`IADRoot` and want to *validate*
that the active doc is the expected type. Each raises `RuntimeError`
if the active session is the wrong type.

> **Prereq for this cell:** open any part.

In [ ]:
from alibrex import connect, require_active_part

In [ ]:
root = connect()
part = require_active_part(root)
part.Name

The full set of strict checkers:

| Function | Returns | Raises if active doc is… |
|---|---|---|
| `require_active_part(root)` | `IADPartSession` | not a part / sheet-metal |
| `require_active_assembly(root)` | `IADAssemblySession` | not an assembly |
| `require_active_drawing(root)` | `IADDrawingSession` | not a drawing |

## 7. `reset_alibre()`: re-bind after Alibre restart

The bound root is cached at module level for the lifetime of the
Python process. If you **close and reopen** Alibre (or restart it),
the cached proxy points at the dead process. Call `reset_alibre()`
to drop the cache so the next `connect()` / `CurrentPart()` binds
the fresh instance.

In [ ]:
from alibrex import reset_alibre

In [ ]:
reset_alibre()
root = connect()
root.Version

## 8. Bypass the import-time check: `ALIBREX_SKIP_RUNNING_CHECK=1`

Set this environment variable **before** importing alibrex to skip
the running-Alibre check at import time. Intended for documentation
builds, stub generation, and CI tooling that doesn't have Alibre
available. `connect()` / `CurrentPart()` still require a real Alibre,
though you can introspect the typed surface without one.

In [ ]:
import os
os.environ.get("ALIBREX_SKIP_RUNNING_CHECK", "<unset>")

## Summary

| Method | Returns | When to use |
|---|---|---|
| `import alibrex` | module | Always - implicit running check |
| `connect()` | `IADRoot` | Shortest path to the root |
| `connect_to_running_alibre()` | `IADRoot` | Same, longer/explicit name |
| `CurrentPart()` | `IADPartSession` | Active doc is a part |
| `CurrentAssembly()` | `IADAssemblySession` | Active doc is an assembly |
| `require_active_part(root)` | `IADPartSession` | Fail-fast validation |
| `require_active_assembly(root)` | `IADAssemblySession` | Fail-fast validation |
| `require_active_drawing(root)` | `IADDrawingSession` | Fail-fast validation |
| `reset_alibre()` | `None` | After closing/reopening Alibre |